In [ ]:
# -*- coding: utf-8 -*-
"""
LiNGAM 7-panels figure per region:
a-f: Top6 feature boxplots (Disaster=1 vs 0)
g: LiNGAM |Total effect| bar + inset (group means, overlap allowed)
PNG+SVG, Times New Roman 9pt, svg font kept as text.
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from lingam import DirectLiNGAM

# =========================
# 0) Style
# =========================
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# =========================
# 1) Paths
# =========================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_path = os.path.join(base_path, "data")
path_name = "Positive_Negative_balanced"
print(path_name)

# ✅ as you requested
out_dir = os.path.join(base_path, "output", path_name, "lingam", "division")
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)

df_path = os.path.join(
    data_path, "Extracted_HAVE_future", "Positive_Negative_balanced",
    "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned_division.csv"
)
if not os.path.exists(df_path):
    alt = os.path.join(os.getcwd(), "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned_division.csv")
    if os.path.exists(alt):
        df_path = alt

df = pd.read_csv(df_path)
print("Loaded:", df.shape, "from", df_path)

# =========================
# 2) Columns & features
# =========================
TARGET_COL = "Disaster"
DIV_COL = "DIV_EN"

FEATURES = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
    "UrbanFrac_hist_2000_2010_2020",
    "PopTotal_hist_2000_2010_2020",
    "ImperviousIndex_hist_2000_2010_2020",
    "LAI_hist_2000_2010_2020",
    "Precip_hist_2000_2010_2020",
    "WTD_hist_2000_2010_2020",
    "HDS_hist_2000_2010_2020",
]
ABBR = {
    "Distance_to_karst": "DK",
    "Depth_to_Bedrock": "DB",
    "Distance_to_Fault_m": "DF",
    "UrbanFrac_hist_2000_2010_2020": "UF",
    "PopTotal_hist_2000_2010_2020": "PT",
    "ImperviousIndex_hist_2000_2010_2020": "IP",
    "LAI_hist_2000_2010_2020": "LAI",
    "Precip_hist_2000_2010_2020": "PR",
    "WTD_hist_2000_2010_2020": "WTD",
    "HDS_hist_2000_2010_2020": "HDS",
}

# =========================
# 3) Groups (overlap allowed) with short labels
# =========================
G1 = "Anthro"
G2 = "Hydro\nEnv"
G3 = "Geo"

GROUPS_BY_ABBR = {
    "UF":  [G1],
    "PT":  [G1],
    "IP":  [G1, G2],
    "WTD": [G1, G2],
    "HDS": [G1, G2],
    "PR":  [G2],
    "LAI": [G2],
    "DK":  [G3],
    "DB":  [G3],
    "DF":  [G3],
}

# =========================
# 4) Colors (match your reference)
# =========================
COLOR_INC = "#C6464A"
COLOR_DEC = "#2F5A73"

COL_DARK_RED = "#7A0019"
COL_RED      = "#C1122F"
COL_BEIGE    = "#F3E6C6"
COL_LBLUE    = "#6FA6C8"
COL_DBLUE    = "#0B3D5C"

GROUP_COLOR = {
    G1: COL_DARK_RED,
    G2: COL_RED,
    G3: COL_BEIGE,
    "Mixed": COL_LBLUE,
    "Other": COL_DBLUE,
}

# =========================
# 5) Regions
# =========================
REGIONS_7 = [
    "Northeast China",
    "Northwest China",
    "North China",
    "Central China",
    "Southwest China",
    "South China",
    "East China",
]
REGIONS = REGIONS_7 + ["National"]

# =========================
# Utils
# =========================
def safe_name(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s

def save_png_svg(fig, out_folder, stem, dpi=300):
    os.makedirs(out_folder, exist_ok=True)
    fig.savefig(os.path.join(out_folder, f"{stem}.png"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(out_folder, f"{stem}.svg"), bbox_inches="tight")
    plt.close(fig)

def groups_for_abbr(a: str):
    return GROUPS_BY_ABBR.get(a, [])

def bar_color_for_groups(groups):
    if len(groups) == 1:
        return GROUP_COLOR.get(groups[0], GROUP_COLOR["Other"])
    if len(groups) >= 2:
        return GROUP_COLOR["Mixed"]
    return GROUP_COLOR["Other"]

def style_axes_box(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1)
    ax.spines["bottom"].set_linewidth(1)
    ax.tick_params(axis="both", which="major", length=3, width=1)

def style_axes_bar(ax):
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_linewidth(1)
    ax.tick_params(axis="both", which="major", length=3, width=1)

def add_panel_letter(ax, letter, x=-0.18, y=1.04, fontsize=13):
    ax.text(x, y, letter, transform=ax.transAxes, fontweight="bold", fontsize=fontsize)

def _iqr(x: np.ndarray) -> float:
    if x.size == 0:
        return np.nan
    q1, q3 = np.percentile(x, [25, 75])
    return float(q3 - q1)

def _overlay_jitter_points(ax, x_center: float, vals: np.ndarray, max_points=200):
    """当箱体塌缩(IQR≈0)但存在非零/非中位数值时，用少量抖动点提示分布。"""
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return
    # 只抽样“非中位数”的点，避免全0时无意义
    med = np.median(vals)
    cand = vals[vals != med]
    if cand.size == 0:
        return
    # 抽样
    if cand.size > max_points:
        rng = np.random.default_rng(42)
        cand = rng.choice(cand, size=max_points, replace=False)
    # x 抖动
    rng = np.random.default_rng(42)
    jitter = rng.uniform(-0.08, 0.08, size=cand.size)
    ax.scatter(np.full(cand.size, x_center) + jitter, cand,
               s=8, c="black", alpha=0.35, linewidths=0)

def boxplot_two_groups(ax, data_inc, data_dec, ylabel):
    """
    Nature-style boxplot:
    - 均值用“实心圆”
    - 均值被 clip 在 whisker 范围内（不会超出箱须）
    - 保留 IQR 塌缩时的抖动点提示
    """

    data_inc = np.asarray(data_inc, dtype=float)
    data_dec = np.asarray(data_dec, dtype=float)

    b = ax.boxplot(
        [data_inc, data_dec],
        labels=["Increased", "Decreased"],
        patch_artist=True,
        showfliers=False,
        widths=0.82,
        showmeans=False,                 # ❌ 不用默认 mean
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
    )

    # 箱体配色
    for patch, c in zip(b["boxes"], [COLOR_INC, COLOR_DEC]):
        patch.set_facecolor(c)
        patch.set_edgecolor("black")
        patch.set_linewidth(0.8)

    ax.set_ylabel(ylabel)
    style_axes_box(ax)

    # ========= 手动计算并绘制 mean（实心圆 + clip） =========
    def draw_mean(ax, x, data, whiskers):
        if data.size == 0:
            return

        mean = np.mean(data)

        # whisker 上下界
        w_low = whiskers[0].get_ydata()[1]
        w_high = whiskers[1].get_ydata()[1]

        mean_clip = np.clip(mean, w_low, w_high)

        ax.scatter(
            x, mean_clip,
            s=22,
            color="black",
            zorder=4
        )

    # whiskers 顺序：[(inc_low, inc_high), (dec_low, dec_high)]
    whiskers = b["whiskers"]
    draw_mean(ax, 1, data_inc, whiskers[0:2])
    draw_mean(ax, 2, data_dec, whiskers[2:4])

    # ========= 顶部均值文字（真实 mean，不是 clip 后） =========
    m1 = np.mean(data_inc) if data_inc.size else np.nan
    m2 = np.mean(data_dec) if data_dec.size else np.nan

    y0, y1 = ax.get_ylim()
    yr = (y1 - y0) if y1 > y0 else 1.0
    ax.set_ylim(y0, y1 + 0.18 * yr)

    ax.text(1, y1 + 0.04 * yr, f"{m1:.2f}" if np.isfinite(m1) else "NA",
            ha="center", va="bottom")
    ax.text(2, y1 + 0.04 * yr, f"{m2:.2f}" if np.isfinite(m2) else "NA",
            ha="center", va="bottom")

    # ========= IQR 塌缩时的抖动点（保留你的逻辑） =========
    eps = 1e-12
    iqr1 = _iqr(data_inc[np.isfinite(data_inc)])
    iqr2 = _iqr(data_dec[np.isfinite(data_dec)])

    if np.isfinite(iqr1) and iqr1 <= eps:
        _overlay_jitter_points(ax, 1.0, data_inc)
    if np.isfinite(iqr2) and iqr2 <= eps:
        _overlay_jitter_points(ax, 2.0, data_dec)


# =========================
# LiNGAM dense effects (order by LiNGAM, weights by Ridge)
# =========================
def fit_lingam_total_effects_dense(df_region: pd.DataFrame, ridge_alpha=1e-2, jitter_y=0.0):
    cols = FEATURES + [TARGET_COL]
    dat = df_region[cols].to_numpy(dtype=float)

    X_feat = dat[:, :len(FEATURES)]
    y = dat[:, -1]
    if jitter_y > 0:
        rng = np.random.default_rng(42)
        y = y + rng.normal(0, jitter_y, size=y.shape)

    sx = StandardScaler()
    Xs = sx.fit_transform(X_feat)

    sy = StandardScaler()
    ys = sy.fit_transform(y.reshape(-1, 1)).ravel()

    # order on features only
    lingam_feat = DirectLiNGAM()
    lingam_feat.fit(Xs)
    order_feat = list(getattr(lingam_feat, "causal_order_", []))
    if len(order_feat) != Xs.shape[1]:
        order_feat = list(range(Xs.shape[1]))

    p_feat = Xs.shape[1]
    p = p_feat + 1
    B_full = np.zeros((p, p), dtype=float)

    reg = Ridge(alpha=ridge_alpha, fit_intercept=False, random_state=42)

    # estimate feature-feature edges according to order
    for k in range(p_feat):
        idx_cur = int(order_feat[k])
        preds = [int(i) for i in order_feat[:k]]
        if len(preds) == 0:
            continue
        reg.fit(Xs[:, preds], Xs[:, idx_cur])
        for j_local, j_feat in enumerate(preds):
            B_full[idx_cur, j_feat] = reg.coef_[j_local]

    # force Disaster as sink: Y regressed on all features, no Y->X
    reg_y = Ridge(alpha=ridge_alpha, fit_intercept=False, random_state=42)
    reg_y.fit(Xs, ys)
    for j_feat in range(p_feat):
        B_full[p_feat, j_feat] = reg_y.coef_[j_feat]

    I = np.eye(p)
    try:
        T = np.linalg.inv(I - B_full)
    except np.linalg.LinAlgError:
        T = np.linalg.pinv(I - B_full)

    te = T[p_feat, :p_feat]
    abs_te = np.abs(te)
    signs = ["+" if v > 0 else "-" for v in te]
    return abs_te, te, signs, B_full, order_feat

# =========================
# Main per-region plot
# =========================
def run_one_region(df_all, region_name: str):
    if region_name == "National":
        df_region = df_all.loc[df_all[DIV_COL].isin(REGIONS_7)].copy()
    else:
        df_region = df_all.loc[df_all[DIV_COL] == region_name].copy()

    use_cols = [DIV_COL, TARGET_COL] + FEATURES
    for c in use_cols:
        if c not in df_region.columns:
            raise ValueError(f"Missing column: {c}")

    df_region = df_region[use_cols].replace([np.inf, -np.inf], np.nan).dropna()
    if df_region[TARGET_COL].nunique() < 2:
        print(f"[Skip] {region_name}: Disaster has only one class.")
        return

    region_tag = safe_name(region_name)
    region_out = os.path.join(out_dir, region_tag)
    os.makedirs(region_out, exist_ok=True)

    abs_te, te, signs, B_full, order_feat = fit_lingam_total_effects_dense(
        df_region, ridge_alpha=1e-2, jitter_y=0.0
    )

    imp = pd.DataFrame({
        "Feature_full": FEATURES,
        "Feature": [ABBR[f] for f in FEATURES],
        "Abs_total_effect": abs_te,
        "Total_effect": te,
        "Sign": signs,
    })
    imp["Groups"] = imp["Feature"].apply(groups_for_abbr)
    imp["Groups_str"] = imp["Groups"].apply(lambda lst: ";".join(lst))
    imp = imp.sort_values("Abs_total_effect", ascending=False)

    imp.to_csv(os.path.join(region_out, f"{region_tag}_lingam_total_effect_table.csv"),
               index=False, encoding="utf-8-sig")
    np.savetxt(os.path.join(region_out, f"{region_tag}_lingam_adjacency_matrix_B.csv"),
               B_full, delimiter=",", fmt="%.10g")
    with open(os.path.join(region_out, f"{region_tag}_lingam_feature_order.txt"), "w", encoding="utf-8") as f:
        f.write("Feature causal order (0-based index among FEATURES):\n")
        f.write(str(order_feat) + "\n")

    top6 = imp.head(6).copy()

    # ===== figure layout =====
    fig = plt.figure(figsize=(7.1, 6.8))
    gs = GridSpec(nrows=3, ncols=3, height_ratios=[1, 1, 1.35],
                  hspace=0.45, wspace=0.55, figure=fig)

    # a-f
    letters = list("abcdef")
    for i in range(6):
        r = 0 if i < 3 else 1
        c = i % 3
        ax = fig.add_subplot(gs[r, c])

        feat_full = top6.iloc[i]["Feature_full"]
        feat_abbr = top6.iloc[i]["Feature"]

        inc = df_region.loc[df_region[TARGET_COL] == 1, feat_full].dropna().values
        dec = df_region.loc[df_region[TARGET_COL] == 0, feat_full].dropna().values

        boxplot_two_groups(ax, inc, dec, ylabel=feat_abbr)
        add_panel_letter(ax, letters[i], x=-0.18, y=1.04, fontsize=13)

    # g bar
    axg = fig.add_subplot(gs[2, :])
    add_panel_letter(axg, "g", x=-0.10, y=1.02, fontsize=13)

    plot_df = imp.copy().sort_values("Abs_total_effect", ascending=False)
    x = np.arange(len(plot_df))
    colors = [bar_color_for_groups(g) for g in plot_df["Groups"].values]

    bar_w_g = 0.55
    bars = axg.bar(x, plot_df["Abs_total_effect"].values, width=bar_w_g)
    for rect, col in zip(bars, colors):
        rect.set_facecolor(col)
        rect.set_edgecolor("black")
        rect.set_linewidth(0.6)

    axg.set_ylabel("|Total effect| (LiNGAM)")
    axg.set_xticks(x)
    axg.set_xticklabels(plot_df["Feature"].values, rotation=90, ha="center")
    style_axes_bar(axg)

    ymax = float(plot_df["Abs_total_effect"].max()) if len(plot_df) else 1.0
    for i, (v_abs, sgn) in enumerate(zip(plot_df["Abs_total_effect"].values, plot_df["Sign"].values)):
        axg.text(i, v_abs + 0.03 * ymax, sgn, ha="center", va="bottom")
    axg.set_ylim(0, ymax * 1.60)

    # inset (no "H")
    axh = axg.inset_axes([0.62, 0.55, 0.34, 0.38])
    group_order = [G1, G2, G3]
    group_means = []
    for g in group_order:
        mask = plot_df["Groups"].apply(lambda lst: g in lst)
        group_means.append(plot_df.loc[mask, "Abs_total_effect"].mean() if mask.any() else 0.0)

    xi = np.arange(len(group_order))
    bar_w_h = 0.45
    b2 = axh.bar(xi, group_means, width=bar_w_h)
    for rect, g in zip(b2, group_order):
        rect.set_facecolor(GROUP_COLOR[g])
        rect.set_edgecolor("black")
        rect.set_linewidth(0.6)

    axh.set_xticks(xi)
    axh.set_xticklabels(group_order, rotation=90, ha="center")
    axh.tick_params(axis="both", which="major", length=2, width=0.8)
    for sp in axh.spines.values():
        sp.set_linewidth(1)

    # no legend for g
    fig.suptitle(region_name, y=0.995, fontweight="bold")

    stem = f"{region_tag}_Fig_Box_LiNGAM_7panels"
    save_png_svg(fig, region_out, stem)

def main():
    for r in REGIONS:
        run_one_region(df, r)
    print("\nDone. Outputs saved under:", out_dir)

if __name__ == "__main__":
    main()


national的单独拎出来

In [ ]:
# -*- coding: utf-8 -*-
"""
LiNGAM 7-panels figure per region
a–f: boxplots
g: |Total effect| bar
h: inset (group-wise mean)
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from lingam import DirectLiNGAM

# =========================
# Style
# =========================
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

# =========================
# Paths
# =========================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_path = os.path.join(base_path, "data")
out_dir = os.path.join(base_path, "output", "Positive_Negative_balanced", "lingam", "division")
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(os.path.join(
    data_path,
    "Extracted_HAVE_future",
    "Positive_Negative_balanced",
    "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned_division.csv"
))

# =========================
# Columns
# =========================
TARGET_COL = "Disaster"
DIV_COL = "DIV_EN"

FEATURES = [
    "Distance_to_karst", "Depth_to_Bedrock", "Distance_to_Fault_m",
    "UrbanFrac_hist_2000_2010_2020", "PopTotal_hist_2000_2010_2020",
    "ImperviousIndex_hist_2000_2010_2020", "LAI_hist_2000_2010_2020",
    "Precip_hist_2000_2010_2020", "WTD_hist_2000_2010_2020",
    "HDS_hist_2000_2010_2020",
]

ABBR = {
    "Distance_to_karst": "DK", "Depth_to_Bedrock": "DB", "Distance_to_Fault_m": "DF",
    "UrbanFrac_hist_2000_2010_2020": "UF", "PopTotal_hist_2000_2010_2020": "PT",
    "ImperviousIndex_hist_2000_2010_2020": "IP", "LAI_hist_2000_2010_2020": "LAI",
    "Precip_hist_2000_2010_2020": "PR", "WTD_hist_2000_2010_2020": "WTD",
    "HDS_hist_2000_2010_2020": "HDS",
}

# =========================
# Groups
# =========================
G1, G2, G3 = "Anthro", "Hydro\nEnv", "Geo"

GROUPS_BY_ABBR = {
    "UF": [G1], "PT": [G1], "IP": [G1, G2], "WTD": [G1, G2],
    "HDS": [G1, G2], "PR": [G2], "LAI": [G2],
    "DK": [G3], "DB": [G3], "DF": [G3],
}

GROUP_COLOR = {
    G1: "#7A0019",
    G2: "#C1122F",
    G3: "#F3E6C6",
    "Mixed": "#6FA6C8",
    "Other": "#0B3D5C",
}

# =========================
# Utils
# =========================
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "", s.replace(" ", "_"))

def save_png_svg(fig, folder, stem):
    os.makedirs(folder, exist_ok=True)
    fig.savefig(os.path.join(folder, f"{stem}.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(folder, f"{stem}.svg"), bbox_inches="tight")
    plt.close(fig)

def groups_for_abbr(a):
    return GROUPS_BY_ABBR.get(a, [])

def bar_color_for_groups(gs):
    if len(gs) == 1:
        return GROUP_COLOR[gs[0]]
    if len(gs) >= 2:
        return GROUP_COLOR["Mixed"]
    return GROUP_COLOR["Other"]

# =========================
# LiNGAM
# =========================
def fit_lingam(df_region):
    X = StandardScaler().fit_transform(df_region[FEATURES])
    y = StandardScaler().fit_transform(df_region[[TARGET_COL]]).ravel()

    model = DirectLiNGAM()
    model.fit(X)
    order = model.causal_order_

    p = X.shape[1]
    B = np.zeros((p + 1, p + 1))
    reg = Ridge(alpha=1e-2, fit_intercept=False)

    for k, i in enumerate(order):
        parents = order[:k]
        if parents:
            reg.fit(X[:, parents], X[:, i])
            B[i, parents] = reg.coef_

    reg.fit(X, y)
    B[p, :p] = reg.coef_

    T = np.linalg.pinv(np.eye(p + 1) - B)
    te = T[p, :p]

    return np.abs(te), ["+" if v > 0 else "-" for v in te]

# =========================
# Plotting
# =========================
def plot_region(df_region, region_name):
    region_tag = safe_name(region_name)
    out = os.path.join(out_dir, region_tag)

    abs_te, signs = fit_lingam(df_region)

    imp = pd.DataFrame({
        "Feature_full": FEATURES,
        "Feature": [ABBR[f] for f in FEATURES],
        "Abs_total_effect": abs_te,
        "Sign": signs,
    })
    imp["Groups"] = imp["Feature"].apply(groups_for_abbr)
    imp = imp.sort_values("Abs_total_effect", ascending=False)

    fig = plt.figure(figsize=(7.1, 6.8))
    gs = GridSpec(3, 3, height_ratios=[1, 1, 1.35], hspace=0.45, wspace=0.55)

    # a–f
    for i in range(6):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        f = imp.iloc[i]["Feature_full"]
        inc = df_region[df_region[TARGET_COL] == 1][f]
        dec = df_region[df_region[TARGET_COL] == 0][f]
        ax.boxplot([inc, dec], labels=["Increased", "Decreased"],
                   patch_artist=True, showfliers=False)
        ax.set_ylabel(imp.iloc[i]["Feature"])

    # g
    axg = fig.add_subplot(gs[2, :])
    x = np.arange(len(imp))
    bar_w_g = 0.42  # ✅ 更瘦
    bars = axg.bar(
        x, imp["Abs_total_effect"],
        width=bar_w_g,
        color=[bar_color_for_groups(g) for g in imp["Groups"]],
        edgecolor="black", linewidth=0.6
    )
    axg.set_ylabel("|Total effect| (LiNGAM)")
    axg.set_xticks(x)
    axg.set_xticklabels(imp["Feature"], rotation=90)

    ymax = imp["Abs_total_effect"].max()
    for i, s in enumerate(imp["Sign"]):
        axg.text(i, imp.iloc[i]["Abs_total_effect"] + 0.03 * ymax,
                 s, ha="center", va="bottom")

    axg.set_ylim(0, ymax * 1.5)  # ✅ 刻度范围

    # h (inset)
    axh = axg.inset_axes([0.62, 0.62, 0.34, 0.38])  # ✅ 上移
    group_order = [G1, G2, G3]
    means = [
        imp.loc[imp["Groups"].apply(lambda x: g in x), "Abs_total_effect"].mean()
        for g in group_order
    ]

    bar_w_h = 0.32  # ✅ 更瘦
    axh.bar(
        np.arange(3), means,
        width=bar_w_h,
        color=[GROUP_COLOR[g] for g in group_order],
        edgecolor="black", linewidth=0.6
    )
    axh.set_xticks(np.arange(3))
    axh.set_xticklabels(group_order, rotation=90)
    for sp in axh.spines.values():
        sp.set_linewidth(1)

    fig.suptitle(region_name, fontweight="bold")
    save_png_svg(fig, out, f"{region_tag}_Fig_Box_LiNGAM_7panels")

# =========================
# Run
# =========================
REGIONS_7 = [
    "Northeast China", "Northwest China", "North China",
    "Central China", "Southwest China", "South China", "East China"
]

def main():
    for r in REGIONS_7:
        plot_region(df[df[DIV_COL] == r].dropna(), r)

    plot_region(df[df[DIV_COL].isin(REGIONS_7)].dropna(), "National")

if __name__ == "__main__":
    main()


全国范围的G图和h图改为累计条形图

In [ ]:
# -*- coding: utf-8 -*-
"""
LiNGAM 7-panels figure per region
a–f: boxplots
g: |Total effect| (National: stacked by 7 regions, 10 features)
h: inset (National: stacked by 7 regions, 3 feature groups)
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from lingam import DirectLiNGAM

# =========================
# Style
# =========================
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

# =========================
# Paths
# =========================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_path = os.path.join(base_path, "data")
out_dir = os.path.join(base_path, "output", "Positive_Negative_balanced", "lingam", "division")
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(os.path.join(
    data_path,
    "Extracted_HAVE_future",
    "Positive_Negative_balanced",
    "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned_division.csv"
))

# =========================
# Columns
# =========================
TARGET_COL = "Disaster"
DIV_COL = "DIV_EN"

FEATURES = [
    "Distance_to_karst", "Depth_to_Bedrock", "Distance_to_Fault_m",
    "UrbanFrac_hist_2000_2010_2020", "PopTotal_hist_2000_2010_2020",
    "ImperviousIndex_hist_2000_2010_2020", "LAI_hist_2000_2010_2020",
    "Precip_hist_2000_2010_2020", "WTD_hist_2000_2010_2020",
    "HDS_hist_2000_2010_2020",
]

ABBR = {
    "Distance_to_karst": "DK", "Depth_to_Bedrock": "DB", "Distance_to_Fault_m": "DF",
    "UrbanFrac_hist_2000_2010_2020": "UF", "PopTotal_hist_2000_2010_2020": "PT",
    "ImperviousIndex_hist_2000_2010_2020": "IP", "LAI_hist_2000_2010_2020": "LAI",
    "Precip_hist_2000_2010_2020": "PR", "WTD_hist_2000_2010_2020": "WTD",
    "HDS_hist_2000_2010_2020": "HDS",
}

# =========================
# Groups
# =========================
G1, G2, G3 = "Anthro", "Hydro\nEnv", "Geo"

GROUPS_BY_ABBR = {
    "UF": [G1], "PT": [G1], "IP": [G1, G2], "WTD": [G1, G2],
    "HDS": [G1, G2], "PR": [G2], "LAI": [G2],
    "DK": [G3], "DB": [G3], "DF": [G3],
}

GROUP_ORDER = [G1, G2, G3]

# =========================
# Region colors
# =========================
REGION_COLORS = {
    "Northeast China": "#2F5A73",
    "North China":     "#5F7F91",
    "East China":      "#9FB6C2",
    "Central China":   "#C6464A",
    "South China":     "#D97C6A",
    "Southwest China": "#E6B3A4",
    "Northwest China": "#9E9E9E",
}

REGIONS_7 = list(REGION_COLORS.keys())

# =========================
# Utils
# =========================
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "", s.replace(" ", "_"))

def save_png_svg(fig, folder, stem):
    fig.savefig(os.path.join(folder, f"{stem}.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(folder, f"{stem}.svg"), bbox_inches="tight")
    plt.close(fig)

# =========================
# LiNGAM
# =========================
def fit_lingam(df_region):
    X = StandardScaler().fit_transform(df_region[FEATURES])
    y = StandardScaler().fit_transform(df_region[[TARGET_COL]]).ravel()

    model = DirectLiNGAM()
    model.fit(X)
    order = model.causal_order_

    p = X.shape[1]
    B = np.zeros((p + 1, p + 1))
    reg = Ridge(alpha=1e-2, fit_intercept=False)

    for k, i in enumerate(order):
        parents = order[:k]
        if parents:
            reg.fit(X[:, parents], X[:, i])
            B[i, parents] = reg.coef_

    reg.fit(X, y)
    B[p, :p] = reg.coef_

    T = np.linalg.pinv(np.eye(p + 1) - B)
    return np.abs(T[p, :p])

# =========================
# Plotting
# =========================
def plot_region(df_region, region_name):
    region_tag = safe_name(region_name)
    out = os.path.join(out_dir, region_tag)

    te_nat = fit_lingam(df_region)

    imp = pd.DataFrame({
        "Feature_full": FEATURES,
        "Feature": [ABBR[f] for f in FEATURES],
        "Abs_total_effect": te_nat,
    })
    imp["Group"] = imp["Feature"].apply(lambda x: GROUPS_BY_ABBR[x][0])
    imp = imp.sort_values("Abs_total_effect", ascending=False)

    # collect regional TEs only for National
    regional_te = {}
    if region_name == "National":
        for r in REGIONS_7:
            regional_te[r] = fit_lingam(df[df[DIV_COL] == r].dropna())

    fig = plt.figure(figsize=(7.1, 6.8))
    gs = GridSpec(3, 3, height_ratios=[1, 1, 1.35], hspace=0.45, wspace=0.55)

    # a–f boxplots
    for i in range(6):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        f = imp.iloc[i]["Feature_full"]
        inc = df_region[df_region[TARGET_COL] == 1][f]
        dec = df_region[df_region[TARGET_COL] == 0][f]
        ax.boxplot([inc, dec], labels=["Increased", "Decreased"],
                   patch_artist=True, showfliers=False)
        ax.set_ylabel(imp.iloc[i]["Feature"])

    # ================= g =================
    axg = fig.add_subplot(gs[2, :])
    x = np.arange(len(imp))
    width = 0.42

    if region_name != "National":
        axg.bar(x, imp["Abs_total_effect"], width=width)
    else:
        bottom = np.zeros(len(imp))
        for r in REGIONS_7:
            contrib = []
            for i, f in enumerate(imp["Feature_full"]):
                idx = FEATURES.index(f)
                denom = sum(regional_te[k][idx] for k in REGIONS_7)
                contrib.append(
                    imp.iloc[i]["Abs_total_effect"] * regional_te[r][idx] / denom
                    if denom > 0 else 0
                )
            axg.bar(x, contrib, bottom=bottom, width=width,
                    color=REGION_COLORS[r], edgecolor=None, linewidth=0, label=r)
            bottom += np.array(contrib)

        axg.legend(ncol=4, frameon=False,
                   loc="lower center", bbox_to_anchor=(0.5, 1.02))

    axg.set_ylabel("|Total effect| (LiNGAM)")
    axg.set_xticks(x)
    axg.set_xticklabels(imp["Feature"], rotation=90)

    # ================= h (group-level stacked) =================
    axh = axg.inset_axes([0.62, 0.62, 0.34, 0.38])

    if region_name != "National":
        means = [
            imp.loc[imp["Group"] == g, "Abs_total_effect"].mean()
            for g in GROUP_ORDER
        ]
        axh.bar(np.arange(3), means, width=0.32)
    else:
        bottom = np.zeros(3)
        for r in REGIONS_7:
            vals = []
            for g in GROUP_ORDER:
                idxs = imp.index[imp["Group"] == g].tolist()
                num = sum(regional_te[r][FEATURES.index(imp.loc[i, "Feature_full"])]
                          for i in idxs)
                den = sum(sum(regional_te[k][FEATURES.index(imp.loc[i, "Feature_full"])]
                              for i in idxs) for k in REGIONS_7)
                vals.append(
                    np.mean(imp.loc[idxs, "Abs_total_effect"]) * num / den
                    if den > 0 else 0
                )
            axh.bar(np.arange(3), vals, bottom=bottom, width=0.32,
                    color=REGION_COLORS[r], edgecolor=None, linewidth=0)
            bottom += np.array(vals)

    axh.set_xticks(np.arange(3))
    axh.set_xticklabels(GROUP_ORDER, rotation=90)

    fig.suptitle(region_name, fontweight="bold")
    save_png_svg(fig, out, f"{region_tag}_Fig_Box_LiNGAM_7panels")

# =========================
# Run
# =========================
def main():
    for r in REGIONS_7:
        plot_region(df[df[DIV_COL] == r].dropna(), r)
    plot_region(df[df[DIV_COL].isin(REGIONS_7)].dropna(), "National")

if __name__ == "__main__":
    main()
